# 8. Visualise PyNNLF Output: SA BESS 44hh Clean Cohort

Databricks full-run version. This notebook visualises the processed CSVs produced from the complete `experiment_result_databricks` run.


## 1. Setup And Paths

In [0]:
from pathlib import Path
import sys


def find_publication_project(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "specs").exists() and (candidate / "data").exists() and (candidate / "results").exists():
            return candidate
    raise FileNotFoundError("Could not find publication/journal_article_1 from the current working directory.")


def find_repo_root(start: Path) -> Path:
    for candidate in [start.resolve(), *start.resolve().parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "src" / "pynnlf").exists():
            return candidate
    raise FileNotFoundError("Could not find the PyNNLF repo root.")


PROJECT_DIR = find_publication_project()
REPO_ROOT = find_repo_root(PROJECT_DIR)
DATA_DIR = PROJECT_DIR / "data"
RESULTS_DIR = PROJECT_DIR / "results" / "04_sa_bess_clean_44hh"
FIGURES_DIR = RESULTS_DIR / "figures"
EXPERIMENT_SOURCE_DIR = PROJECT_DIR / "experiment_result_databricks"
RESULT_SOURCE_LABEL = "Databricks complete run"

print(f"Publication project: {PROJECT_DIR}")
print(f"Repository root: {REPO_ROOT}")
print(f"Experiment source: {EXPERIMENT_SOURCE_DIR}")
print(f"Result source: {RESULT_SOURCE_LABEL}")

import pandas as pd
import matplotlib.pyplot as plt


## 2. Load Datasets

In [0]:
series = []
for key, spec in DATASET_SPECS.items():
    path = DATA_DIR / spec["filename"]
    if not path.exists():
        raise FileNotFoundError(f"Missing dataset {path}. Run notebook 5 first.")
    df = pd.read_csv(path, parse_dates=["datetime"])
    if len(df) != 17_520 or df.isna().any().any():
        raise ValueError(f"{path.name} failed basic validation")
    series.append(df.rename(columns={"netload_kW": key}))
datasets = series[0]
for frame in series[1:]:
    datasets = datasets.merge(frame, on="datetime", how="inner")
if datasets.shape[0] != 17_520:
    raise ValueError(f"Merged dataset expected 17,520 rows, found {datasets.shape[0]:,}")
display(datasets.head())

## 3. Dataset Profiles And Distributions

In [0]:
summary_stats = datasets[DATASET_ORDER].agg(["mean", "std", "min", "median", "max"]).T.rename(index=DATASET_LABELS).round(3)
display(summary_stats)
fig, ax = plt.subplots(figsize=(8, 5))
datasets[DATASET_ORDER].rename(columns=DATASET_LABELS).plot.box(ax=ax)
ax.set_ylabel("Aggregate load (kW)")
ax.set_title("SA BESS 44-household annual half-hourly load distributions")
ax.tick_params(axis="x", rotation=12)
fig.tight_layout()
save_figure(fig, "fig01_dataset_distributions.png")
plt.show()
profile_source = datasets.copy()
profile_source["week_slot"] = profile_source["datetime"].dt.dayofweek * 48 + profile_source["datetime"].dt.hour * 2 + profile_source["datetime"].dt.minute // 30
typical_week = profile_source.groupby("week_slot")[DATASET_ORDER].mean().reindex(range(7 * 48))
fig, ax = plt.subplots(figsize=(13, 5))
for dataset_key in DATASET_ORDER:
    ax.plot(typical_week.index, typical_week[dataset_key], label=DATASET_LABELS[dataset_key], color=COLORS[dataset_key], linewidth=2.0)
for day in range(8):
    ax.axvline(day * 48, color="0.85", linewidth=0.8, zorder=0)
ax.set_xticks([day * 48 + 24 for day in range(7)])
ax.set_xticklabels(["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"])
ax.set_ylabel("Aggregate load (kW)")
ax.set_title("Typical weekly profile of SA BESS 44-household clean datasets")
ax.legend(loc="upper left")
fig.tight_layout()
save_figure(fig, "fig02_typical_week_profile.png")
plt.show()

## 4. Load Databricks-Complete PyNNLF Result Tables

In [0]:
result_files = {"nrmse": RESULTS_DIR / "sa_bess_44hh_fh8_nrmse_comparison.csv", "stddev": RESULTS_DIR / "sa_bess_44hh_fh8_nrmse_stddev_comparison.csv"}
missing = [str(path) for path in result_files.values() if not path.exists()]
if missing:
    raise FileNotFoundError("Missing processed PyNNLF result files. Run notebook 7 after the Databricks complete experiments. " + str(missing))
nrmse = pd.read_csv(result_files["nrmse"], index_col=0).reindex(MODEL_ORDER)
stddev = pd.read_csv(result_files["stddev"], index_col=0).reindex(MODEL_ORDER)
display(nrmse.round(3))

## 5. Model Performance Figure

In [0]:
fig, ax = plt.subplots(figsize=(14, 6))
nrmse.T.rename(index=DATASET_LABELS).plot(kind="bar", yerr=stddev.T, ax=ax, width=0.82, capsize=2)
ax.set_ylabel("Test nRMSE (%)")
ax.set_xlabel("")
ax.set_title("SA BESS 44-household clean cohort: 1-day model performance")
ax.tick_params(axis="x", rotation=0)
ax.legend(title="Model / hyperparameter", bbox_to_anchor=(1.02, 1), loc="upper left")
fig.tight_layout()
save_figure(fig, "fig03_model_performance_by_dataset.png")
plt.show()

## 6. Figure Inventory

In [0]:
for path in sorted(FIGURES_DIR.glob("fig*.png")):
    print(path)